# Edit Clustering Score (ECS) vs Jaccard for Near-Duplicate Text Detection

This notebook demonstrates the **Edit Clustering Score (ECS)** — a novel feature for near-duplicate text detection based on the spatial distribution of word-level edits.

**Hypothesis:** When one article is created by splicing a contiguous block from another (near-duplicate), edits are *clustered* in one region, producing a low Index of Dispersion (IoD) of inter-edit gaps. Random or hard-negative pairs have edits *scattered* throughout, producing high IoD.

**Experiment:** 4 classifier variants (jaccard_only, ecs_only, jaccard_ecs, all_features) are evaluated with 5-fold stratified CV on synthetic article pairs (near-dup / hard-neg / random).

**Result (DISCONFIRMED):** 5-gram Jaccard is a perfect ceiling (AUC=1.0) for this synthetic dataset. ECS captures real but redundant signal. Contiguous splices preserve many k-grams, making Jaccard inherently sensitive to edit locality.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru is not pre-installed on Colab
_pip('loguru==0.7.3')

# Core scientific packages — pre-installed on Colab, install locally only
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
import difflib
import gc
import json
import math
import os
import random
import sys
from collections import defaultdict
from typing import Any

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-f2202a-edit-clustering-score-spatial-edit-patte/main/round-1/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded {data['metadata']['n_pairs']} pre-computed pairs")
print(f"Features: {data['metadata']['features']}")
print(f"Pair types: {data['metadata']['pair_types']}")

## Config

All tunable parameters are defined here. Start with minimum values for a quick demo run.
To reproduce the full paper results, use the commented-out values.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Demo (minimum) values — runs in seconds
N_ARTICLES = 100       # full paper: 3000
PAIRS_PER_CLASS = 10   # full paper: 300
MAX_IOD_CLIP = 200.0
MIN_WORDS = 100
MAX_WORDS = 600

## Feature Functions

`jaccard_ngram`: 5-gram Jaccard similarity between two texts (overlap of word 5-shingles).

`compute_ecs`: Edit Clustering Score = Index of Dispersion (var/mean) of inter-edit-gap lengths from a word-level LCS diff. Low IoD means edits are clustered (contiguous splice); high IoD means edits are scattered.

In [ ]:
def jaccard_ngram(t1: str, t2: str, k: int = 5) -> float:
    words1 = t1.lower().split()
    words2 = t2.lower().split()
    s1 = set(tuple(words1[i: i + k]) for i in range(len(words1) - k + 1))
    s2 = set(tuple(words2[i: i + k]) for i in range(len(words2) - k + 1))
    if not s1 or not s2:
        return 0.0
    return len(s1 & s2) / len(s1 | s2)


def compute_ecs(t1: str, t2: str) -> dict[str, float]:
    """ECS = IoD of inter-edit-gap lengths from word-level LCS diff."""
    w1 = t1.lower().split()
    w2 = t2.lower().split()
    total_len = max(len(w1), 1)

    matcher = difflib.SequenceMatcher(None, w1, w2, autojunk=False)
    opcodes = matcher.get_opcodes()

    edit_positions = []
    longest_run = 0
    current_run = 0

    for tag, i1, i2, j1, j2 in opcodes:
        if tag != "equal":
            mid = (i1 + i2) / 2.0
            edit_positions.append(mid)
            current_run += i2 - i1
            longest_run = max(longest_run, current_run)
        else:
            current_run = 0

    n_edits = len(edit_positions)
    edit_count_norm = n_edits / total_len

    edit_span_frac = 0.0
    if n_edits > 1:
        edit_span_frac = (edit_positions[-1] - edit_positions[0]) / total_len

    longest_run_frac = longest_run / total_len

    if n_edits < 2:
        iod = 0.0
    else:
        gaps = np.diff(edit_positions)
        mean_gap = float(np.mean(gaps))
        if mean_gap == 0:
            iod = 0.0
        else:
            iod = float(np.var(gaps) / mean_gap)
    iod = min(iod, MAX_IOD_CLIP)

    return {
        "ecs": iod,
        "edit_count": n_edits,
        "edit_count_norm": edit_count_norm,
        "edit_span_frac": edit_span_frac,
        "longest_run": longest_run_frac,
    }


def compute_features(row: tuple[str, str, int, str]) -> dict[str, Any]:
    t1, t2, label, pair_type = row
    if len(t1.split()) < 10 or len(t2.split()) < 10:
        return {"label": label, "pair_type": pair_type, "skip": True}
    jac = jaccard_ngram(t1, t2)
    ecs_feats = compute_ecs(t1, t2)
    return {"label": label, "pair_type": pair_type, "jaccard": jac, "skip": False, **ecs_feats}

## Dataset Construction

Synthetic articles are generated from 5 topic-specific vocabularies (politics/sports/science/business/technology). Three pair types:
- **near_dup**: article A with a contiguous 20-40% word block spliced from article B
- **hard_neg**: same-category article pairs (similar vocabulary, different content)
- **random**: cross-category pairs (different vocabulary, very different content)

In [ ]:
def load_articles(n: int) -> list[dict]:
    """Generate synthetic articles from topic-specific vocabularies (no network needed)."""
    categories = {
        0: ("politics", [
            "government", "senator", "president", "election", "policy", "congress",
            "democrat", "republican", "vote", "legislation", "campaign", "parliament",
            "minister", "constitution", "democracy", "bill", "committee", "federal",
            "state", "law", "party", "candidate", "ballot", "reform", "debate",
            "administration", "cabinet", "senate", "house", "speaker", "amendment",
            "judiciary", "executive", "regulation", "treaty", "diplomacy", "foreign",
            "domestic", "budget", "taxation", "healthcare", "immigration", "security",
            "military", "defense", "intelligence", "sanctions", "coalition", "majority",
            "minority", "opposition", "leadership", "summit", "agreement", "resolution",
            "veto", "impeachment", "scandal", "investigation", "testimony", "hearing",
        ]),
        1: ("sports", [
            "football", "soccer", "basketball", "baseball", "tennis", "championship",
            "athlete", "stadium", "tournament", "coach", "team", "player", "score",
            "goal", "match", "season", "league", "trophy", "medal", "training",
            "defender", "midfielder", "striker", "goalkeeper", "referee", "penalty",
            "tackle", "dribble", "sprint", "marathon", "swimming", "cycling", "rowing",
            "gymnastics", "boxing", "wrestling", "skiing", "skating", "volleyball",
            "cricket", "rugby", "polo", "golf", "victory", "defeat", "draw",
            "semifinal", "qualifier", "ranking", "transfer", "contract", "sponsor",
            "broadcast", "spectator", "fan", "arena", "pitch", "court", "track",
        ]),
        2: ("science", [
            "research", "experiment", "hypothesis", "laboratory", "molecule", "protein",
            "genome", "evolution", "quantum", "particle", "photon", "electron",
            "neuron", "antibody", "vaccine", "pathogen", "climate", "ecosystem",
            "biodiversity", "taxonomy", "astronomy", "telescope", "galaxy", "asteroid",
            "orbit", "gravity", "radiation", "isotope", "catalyst", "polymer",
            "semiconductor", "algorithm", "computation", "simulation", "dataset",
            "neural", "statistical", "empirical", "methodology", "analysis", "synthesis",
            "compound", "reaction", "entropy", "thermodynamics", "magnetism", "optics",
            "microscope", "spectroscopy", "measurement", "observation", "theory",
            "publication", "journal", "peer", "review", "citation", "discovery",
        ]),
        3: ("business", [
            "market", "revenue", "profit", "investor", "startup", "corporation",
            "merger", "acquisition", "stock", "dividend", "shareholder", "equity",
            "debt", "credit", "banking", "insurance", "commodity", "currency",
            "inflation", "recession", "growth", "gdp", "trade", "export", "import",
            "tariff", "supply", "demand", "consumer", "retail", "wholesale", "logistics",
            "manufacturing", "production", "factory", "outsourcing", "franchise",
            "brand", "marketing", "advertising", "campaign", "launch", "product",
            "service", "customer", "subscription", "pricing", "discount", "quarterly",
            "annual", "forecast", "earnings", "valuation", "funding", "venture",
            "capital", "portfolio", "hedge", "futures", "derivative", "commodity",
        ]),
        4: ("technology", [
            "software", "hardware", "processor", "memory", "network", "internet",
            "cloud", "server", "database", "encryption", "cybersecurity", "hacking",
            "blockchain", "cryptocurrency", "artificial", "intelligence", "machine",
            "learning", "algorithm", "neural", "robot", "automation", "sensor",
            "device", "smartphone", "operating", "system", "application", "platform",
            "interface", "bandwidth", "protocol", "wireless", "fiber", "latency",
            "compiler", "framework", "library", "api", "microservice", "container",
            "virtualization", "quantum", "computing", "transistor", "chip", "silicon",
            "battery", "renewable", "satellite", "streaming", "codec", "resolution",
            "pixel", "display", "camera", "printer", "storage", "backup", "recovery",
        ]),
    }
    common_words = [
        "the", "and", "that", "this", "with", "from", "have", "been", "will",
        "more", "also", "after", "some", "their", "when", "which", "said",
        "over", "such", "into", "than", "other", "could", "about", "first",
        "time", "year", "new", "last", "long", "make", "many", "well", "only",
        "two", "may", "use", "even", "most", "both", "very", "each", "where",
    ]

    rng = random.Random(SEED + 1)
    articles = []
    n_cats = len(categories)
    for i in range(n):
        cat_id = i % n_cats
        cat_name, cat_words = categories[cat_id]
        length = rng.randint(280, 340)
        words = []
        for _ in range(length):
            if rng.random() < 0.78:
                words.append(rng.choice(cat_words))
            else:
                words.append(rng.choice(common_words))
        text = " ".join(words)
        articles.append({"title": f"{cat_name}_{i}", "text": text, "label": cat_id})

    print(f"Generated {len(articles)} synthetic articles")
    return articles


def make_near_dup(a: dict, b: dict, rng: random.Random) -> tuple[str, str]:
    """Replace a contiguous 20-40% word span of A with words from B."""
    words_a = a["text"].split()
    words_b = b["text"].split()
    n = len(words_a)
    frac = rng.uniform(0.2, 0.4)
    span = max(1, int(n * frac))
    start = rng.randint(0, max(0, n - span))
    replacement = words_b[:span]
    modified = words_a[:start] + replacement + words_a[start + span:]
    return a["text"], " ".join(modified)


def build_pairs(articles: list[dict], pairs_per_class: int, seed: int = SEED) -> list[tuple[str, str, int, str]]:
    rng = random.Random(seed)
    art_list = articles[:]
    rng.shuffle(art_list)

    buckets: dict[Any, list] = defaultdict(list)
    for a in art_list:
        key = a.get("label", a["title"][:4].lower())
        buckets[key].append(a)

    pairs: list[tuple[str, str, int, str]] = []

    # Near-duplicates: localized splice
    i = 0
    while len([p for p in pairs if p[2] == 1]) < pairs_per_class and i < len(art_list) - 1:
        t1, t2 = make_near_dup(art_list[i], art_list[i + 1], rng)
        pairs.append((t1, t2, 1, "near_dup"))
        i += 2

    # Hard negatives: same topic bucket, different articles
    hd_pool = []
    for bucket in buckets.values():
        if len(bucket) >= 2:
            a, b = rng.sample(bucket, 2)
            hd_pool.append((a["text"], b["text"], 0, "hard_neg"))
    rng.shuffle(hd_pool)
    pairs.extend(hd_pool[:pairs_per_class])

    # Random pairs: different articles
    while len([p for p in pairs if p[3] == "random"]) < pairs_per_class:
        a, b = rng.sample(art_list, 2)
        pairs.append((a["text"], b["text"], 0, "random"))

    rng.shuffle(pairs)
    print(f"Built {len(pairs)} pairs total")
    return pairs

## Generate Articles and Build Pairs

Generate synthetic articles and construct pairs. Features are computed sequentially (no multiprocessing for demo simplicity).

In [ ]:
articles = load_articles(N_ARTICLES)
if len(articles) < PAIRS_PER_CLASS * 3:
    print(f"Only {len(articles)} articles, reducing pairs_per_class")
    PAIRS_PER_CLASS = max(5, len(articles) // 6)

pairs = build_pairs(articles, PAIRS_PER_CLASS)
del articles
gc.collect()

# Compute features sequentially (original uses ProcessPoolExecutor)
print(f"Computing features for {len(pairs)} pairs...")
feature_rows = [compute_features(row) for row in pairs]
feature_rows = [r for r in feature_rows if not r.get("skip", False)]
print(f"Features computed for {len(feature_rows)} pairs (skipped {len(pairs)-len(feature_rows)})")

df = pd.DataFrame(feature_rows)
print(f"DataFrame shape: {df.shape}")
print(f"Label distribution: {df['label'].value_counts().to_dict()}")
print(f"Pair type distribution: {df['pair_type'].value_counts().to_dict()}")

## Feature Summaries and Mann-Whitney Test

Check if ECS (IoD) is directionally higher for near-duplicates vs negatives as hypothesized.

In [ ]:
# Quick sanity: check for NaN/inf
n_bad = df[["jaccard", "ecs", "edit_count_norm", "edit_span_frac", "longest_run"]].isnull().sum().sum()
n_inf = np.isinf(df[["jaccard", "ecs", "edit_count_norm", "edit_span_frac", "longest_run"]].values).sum()
if n_bad > 0 or n_inf > 0:
    print(f"Found {n_bad} NaN and {n_inf} inf values — filling with 0")
    df = df.fillna(0)
    df = df.replace([np.inf, -np.inf], 0)

nd = df[df["label"] == 1]
neg = df[df["label"] == 0]
hn = df[df["pair_type"] == "hard_neg"]
rnd = df[df["pair_type"] == "random"]

feature_summary = {
    "median_jaccard_near_dup": float(nd["jaccard"].median()),
    "median_jaccard_hard_neg": float(hn["jaccard"].median()) if len(hn) > 0 else None,
    "median_jaccard_random": float(rnd["jaccard"].median()) if len(rnd) > 0 else None,
    "median_ecs_near_dup": float(nd["ecs"].median()),
    "median_ecs_hard_neg": float(hn["ecs"].median()) if len(hn) > 0 else None,
    "median_ecs_random": float(rnd["ecs"].median()) if len(rnd) > 0 else None,
    "mean_ecs_near_dup": float(nd["ecs"].mean()),
    "mean_ecs_neg": float(neg["ecs"].mean()),
}
print("Feature summary:")
for k, v in feature_summary.items():
    print(f"  {k}: {v:.4f}" if v is not None else f"  {k}: N/A")

# Mann-Whitney on ECS: near-dup vs negatives
iod_nd = nd["ecs"].values
iod_neg = neg["ecs"].values
iod_hn = hn["ecs"].values if len(hn) > 0 else np.array([])

stat_all, pval_all = mannwhitneyu(iod_nd, iod_neg, alternative="greater")
median_ratio_all = float(np.median(iod_nd)) / (float(np.median(iod_neg)) + 1e-9)

mw_hn = {}
if len(iod_hn) > 0:
    stat_hn, pval_hn = mannwhitneyu(iod_nd, iod_hn, alternative="greater")
    mw_hn = {
        "statistic": float(stat_hn),
        "p_value": float(pval_hn),
        "median_iod_near_dup": float(np.median(iod_nd)),
        "median_iod_hard_neg": float(np.median(iod_hn)),
        "median_ratio": float(np.median(iod_nd)) / (float(np.median(iod_hn)) + 1e-9),
    }
    print(f"MW (ND vs HN): p={pval_hn:.4f}, ratio={mw_hn['median_ratio']:.2f}")

print(f"MW (ND vs all-neg): p={pval_all:.4f}, ratio={median_ratio_all:.2f}")

## Classification: 5-fold Stratified CV

Evaluate 4 classifier variants (jaccard_only, ecs_only, jaccard_ecs, all_features) using logistic regression with StandardScaler.

In [ ]:
def evaluate_classifiers(df: pd.DataFrame) -> tuple[dict, dict]:
    feat_sets = {
        "jaccard_only": ["jaccard"],
        "ecs_only": ["ecs"],
        "jaccard_ecs": ["jaccard", "ecs"],
        "all_features": ["jaccard", "ecs", "edit_count_norm", "edit_span_frac", "longest_run"],
    }

    y = df["label"].values
    n_splits = min(5, df["label"].value_counts().min())
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    results: dict[str, Any] = {}
    all_predictions: dict[str, np.ndarray] = {}

    for name, feats in feat_sets.items():
        X = df[feats].values
        aucs = []
        all_proba = np.zeros(len(df))
        for train_idx, val_idx in skf.split(X, y):
            clf = Pipeline([
                ("scaler", StandardScaler()),
                ("lr", LogisticRegression(max_iter=2000, C=1.0)),
            ])
            clf.fit(X[train_idx], y[train_idx])
            proba = clf.predict_proba(X[val_idx])[:, 1]
            if len(np.unique(y[val_idx])) == 2:
                aucs.append(roc_auc_score(y[val_idx], proba))
            all_proba[val_idx] = proba
        results[name] = {
            "auc_mean": float(np.mean(aucs)) if aucs else float("nan"),
            "auc_std": float(np.std(aucs)) if aucs else float("nan"),
            "auc_folds": [float(a) for a in aucs],
        }
        all_predictions[name] = all_proba
        print(f"  {name}: AUC={results[name]['auc_mean']:.4f} ± {results[name]['auc_std']:.4f}")

    return results, all_predictions


def precision_at_recall(y_true: np.ndarray, scores: np.ndarray, recall_target: float = 0.8) -> float:
    from sklearn.metrics import precision_recall_curve
    prec, rec, _ = precision_recall_curve(y_true, scores)
    mask = rec >= recall_target
    if not mask.any():
        return float("nan")
    return float(prec[mask].max())


print("Running 5-fold CV classification...")
if df["label"].nunique() < 2:
    raise RuntimeError("Dataset has only one class")

clf_results, all_predictions = evaluate_classifiers(df)

# Precision@80% recall
y_true = df["label"].values
p80 = {}
for name, proba in all_predictions.items():
    p80[name] = precision_at_recall(y_true, proba, 0.8)
    print(f"  {name}: P@80%R={p80[name]:.4f}")

## Verdict

Compute the final verdict based on delta_AUC, ECS-only AUC, and the Mann-Whitney test.

In [ ]:
delta_auc = clf_results["jaccard_ecs"]["auc_mean"] - clf_results["jaccard_only"]["auc_mean"]
ecs_only_auc = clf_results["ecs_only"]["auc_mean"]
mw_passes = pval_all < 0.05 and median_ratio_all >= 1.5
verdict = (
    "CONFIRMED"
    if (delta_auc >= 0.03 and ecs_only_auc > 0.6 and median_ratio_all >= 2.0 and pval_all < 0.01)
    else "PARTIAL"
    if (delta_auc >= 0.01 and ecs_only_auc > 0.55 and mw_passes)
    else "DISCONFIRMED"
)
print(f"=== VERDICT: {verdict} | delta_AUC={delta_auc:.4f}, ecs_auc={ecs_only_auc:.4f}, MW_p={pval_all:.4f} ===")

## Results Visualization

Compare AUC scores across classifier variants and visualize ECS (IoD) distributions by pair type.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. AUC bar chart
names = list(clf_results.keys())
aucs = [clf_results[n]["auc_mean"] for n in names]
stds = [clf_results[n]["auc_std"] for n in names]
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
axes[0].bar(names, aucs, yerr=stds, capsize=5, color=colors, alpha=0.85)
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel("AUC (5-fold CV)")
axes[0].set_title("Classifier AUC Comparison")
axes[0].set_xticklabels(names, rotation=15, ha="right")
for i, (a, s) in enumerate(zip(aucs, stds)):
    axes[0].text(i, a + s + 0.02, f"{a:.3f}", ha="center", fontsize=9)

# 2. ECS (IoD) distribution by pair type — use pre-computed mini_demo_data
pt_ecs = {"near_dup": [], "hard_neg": [], "random": []}
for ex in data["examples"]:
    pt = ex["pair_type"]
    if pt in pt_ecs:
        pt_ecs[pt].append(ex["ecs"])

pt_colors = {"near_dup": "#55A868", "hard_neg": "#DD8452", "random": "#C44E52"}
for pt, vals in pt_ecs.items():
    if vals:
        axes[1].hist(vals, bins=15, alpha=0.6, label=pt, color=pt_colors[pt])
axes[1].set_xlabel("ECS (IoD of inter-edit gaps)")
axes[1].set_ylabel("Count")
axes[1].set_title("ECS Distribution by Pair Type\n(near_dup has LOWER ECS — inverted!)")
axes[1].legend()

# 3. Jaccard vs ECS scatter — use computed df
scatter_colors = df["pair_type"].map({"near_dup": "#55A868", "hard_neg": "#DD8452", "random": "#C44E52"})
axes[2].scatter(df["jaccard"], df["ecs"], c=scatter_colors, alpha=0.6, s=30)
# Legend proxies
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=pt) for pt, c in pt_colors.items() if pt in df["pair_type"].values]
axes[2].legend(handles=legend_elements)
axes[2].set_xlabel("5-gram Jaccard")
axes[2].set_ylabel("ECS (IoD)")
axes[2].set_title("Jaccard vs ECS by Pair Type")

plt.tight_layout()
plt.savefig("ecs_results.png", dpi=100, bbox_inches="tight")
plt.show()

# Summary table
print("\n=== RESULTS SUMMARY ===")
print(f"{'Classifier':<20} {'AUC':>8} {'±':>4} {'Std':>8}  {'P@80%R':>8}")
print("-" * 54)
for name in clf_results:
    r = clf_results[name]
    p = p80.get(name, float("nan"))
    print(f"{name:<20} {r['auc_mean']:>8.4f}    {r['auc_std']:>8.4f}  {p:>8.4f}")
print(f"\ndelta_AUC (jaccard_ecs - jaccard_only): {delta_auc:+.4f}")
print(f"MW ND vs neg: p={pval_all:.4f}, median_ratio={median_ratio_all:.2f}")
print(f"\nVERDICT: {verdict}")
print("\nKey insight: near_dup has LOWER ECS (IoD) than negatives because a single")
print("contiguous splice produces few clustered edit positions (low variance/mean),")
print("while scattered edits in random pairs produce high IoD. The hypothesis direction")
print("was inverted. Also, 5-gram Jaccard is already a perfect ceiling classifier.")